In [1]:

# 7_cluster_local_level.ipynb
#
# Clusters the synthetic population at national, LA, or MSOA level.
# CLUSTER_LEVEL controls the granularity — change it and re-run.
#
# Each geography unit is loaded individually from parquet (using filters),
# clustered, and its DNA rows appended to a CSV — no full dataset in memory.
#
# USE_GROUPED_CLUSTERING = True  (default)
#   Splits each unit by employment status (GROUPS from config_cluster.py),
#   runs K-Means within each group → "Employed 1", "Retired 2" etc.
#
# USE_GROUPED_CLUSTERING = False
#   Clusters the entire unit population in one pass with MAX_TOTAL_CLUSTERS clusters.
#   row['group'] is set to "All" — the frontend "All" tab shows everything.

import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_paths as _cp
importlib.reload(_cp)
from data_pipeline.config_paths import USE_TEST_DATA, DATA_FOLDER, USE_FOUR_LA_SUBSET, FOUR_LA_CODES
import data_pipeline.config_variables as _cv
import data_pipeline.config_cluster   as _cc
_cv.reload_config_variables()
importlib.reload(_cc)

import pandas as pd
import numpy  as np
from pathlib import Path
from tqdm    import tqdm

import data_pipeline.helpers.normalise as normalise
import data_pipeline.helpers.cluster as cf
importlib.reload(normalise)
importlib.reload(cf)

from data_pipeline.config_variables import (
    CLUSTER_VARS, SUMMARY_VARS, VARIABLE_MAP,
    CATEGORICAL_VARS, CATEGORY_MAPS, CONTINUOUS_VARS,
)
from data_pipeline.config_cluster import WAVE, GROUPS, MAX_TOTAL_CLUSTERS, USE_GROUPED_CLUSTERING

# ── Config ────────────────────────────────────────────────────────────────────
CLUSTER_LEVEL    = "LA"          # "national" | "LA" | "MSOA"
SYNPOP_PARQUET   = f"../{DATA_FOLDER}/6_synthetic_population/synthetic_population.parquet"
OUTPUT_DIR       = Path(f"../{DATA_FOLDER}/7_cluster_local_level")
GEO_CSV          = Path(f"../{DATA_FOLDER}/0_raw/admin_geography_mappings.csv")
MIN_CLUSTER_SIZE = 5

# UNIT_FILTER — restrict which units are clustered.
#   None              → all units (default, ~350 LAs, slow)
#   "london"          → 33 London boroughs only (E09* codes)
#   ["E09000001", …]  → explicit list of codes
# When USE_FOUR_LA_SUBSET (config_paths): four boroughs only; output file stays LA_london_clusters.csv
UNIT_FILTER = list(FOUR_LA_CODES) if USE_FOUR_LA_SUBSET else "london"

_LEVEL_COL = {
    "national": None,
    "LA":        "ladcd",
    "MSOA":      "msoa21cd",
}
if CLUSTER_LEVEL not in _LEVEL_COL:
    raise ValueError(f"CLUSTER_LEVEL must be one of {list(_LEVEL_COL)}")
LEVEL_COL = _LEVEL_COL[CLUSTER_LEVEL]

if not os.path.exists(SYNPOP_PARQUET):
    raise FileNotFoundError(f"{SYNPOP_PARQUET} not found — run 6a_synthetic_population.ipynb first.")

# Derive output filename from filter so runs don't overwrite each other
_filter_tag = (
    ""         if UNIT_FILTER is None else
    "_london"  if UNIT_FILTER == "london" or USE_FOUR_LA_SUBSET else
    "_custom"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / f"{CLUSTER_LEVEL}{_filter_tag}_clusters.csv"

# Always start fresh — avoids stale rows from previous runs with different columns
if OUTPUT_CSV.exists():
    OUTPUT_CSV.unlink()
    print(f"Removed existing {OUTPUT_CSV.name}")

# ── LA name lookup (for display in frontend) ──────────────────────────────────
_la_name_map = {}
if CLUSTER_LEVEL == "LA" and GEO_CSV.exists():
    _geo = pd.read_csv(GEO_CSV, encoding='latin-1', usecols=['ladcd', 'ladnm']).drop_duplicates('ladcd')
    _la_name_map = _geo.set_index('ladcd')['ladnm'].to_dict()
    print(f"Loaded {len(_la_name_map)} LA names from {GEO_CSV.name}")

# ── Feature columns ───────────────────────────────────────────────────────────
# Read schema only to validate feature columns without loading data
import pyarrow.parquet as pq
parquet_cols = pq.read_schema(SYNPOP_PARQUET).names
feature_cols = [f"{WAVE}_{b}" for b in CLUSTER_VARS if f"{WAVE}_{b}" in parquet_cols]
missing_feat = [f"{WAVE}_{b}" for b in CLUSTER_VARS if f"{WAVE}_{b}" not in parquet_cols]
print(f"Feature columns: {len(feature_cols)} present, {len(missing_feat)} missing")
if missing_feat:
    print(f"  Missing: {missing_feat}")

print(f"\nGrouped clustering: {USE_GROUPED_CLUSTERING}")

# ── Employment group → jbstat OHE column map (grouped mode only) ─────────────
_JBSTAT_LOOKUP = {
    "Employed":   f"{WAVE}_jbstat_1",
    "Unemployed": f"{WAVE}_jbstat_3",
    "Retired":    f"{WAVE}_jbstat_4",
    "On leave":   f"{WAVE}_jbstat_5",
    "Student":    f"{WAVE}_jbstat_7",
    "Inactive":   f"{WAVE}_jbstat_8",
}
present_jbstat = set(parquet_cols)
GROUP_COL = {
    gname: (
        _JBSTAT_LOOKUP.get(gname)
        if _JBSTAT_LOOKUP.get(gname) in present_jbstat
        else (None if _JBSTAT_LOOKUP.get(gname) is None else "MISSING")
    )
    for gname in GROUPS
}
if USE_GROUPED_CLUSTERING:
    print(f"\nGroup -> jbstat column (current-wave, mutually exclusive):")
    for g, c in GROUP_COL.items():
        print(f"  {g:20s} -> {c}")

# ── Geography units ───────────────────────────────────────────────────────────
if LEVEL_COL is None:
    unit_ids = ["national"]
else:
    # Read only the geography column to get the full list of units
    all_unit_ids = (
        pd.read_parquet(SYNPOP_PARQUET, columns=[LEVEL_COL])[LEVEL_COL]
        .dropna().unique().tolist()
    )
    all_unit_ids.sort()

    # Apply UNIT_FILTER
    if UNIT_FILTER is None:
        unit_ids = all_unit_ids
    elif UNIT_FILTER == "london":
        unit_ids = [u for u in all_unit_ids if u.startswith("E09")]
    elif isinstance(UNIT_FILTER, list):
        unit_ids = [u for u in all_unit_ids if u in set(UNIT_FILTER)]
    else:
        raise ValueError(f"UNIT_FILTER must be None, 'london', or a list of codes")

_filter_desc = (
    None if UNIT_FILTER is None else
    "london (33 E09 boroughs)" if UNIT_FILTER == "london" else
    f"four-LA subset ({', '.join(UNIT_FILTER)})" if USE_FOUR_LA_SUBSET else
    f"{len(UNIT_FILTER)} explicit codes"
)
print(f"\nClustering at '{CLUSTER_LEVEL}' level: {len(unit_ids)} unit(s)"
      + (f" [filter: {_filter_desc}]" if _filter_desc else ""))
print(f"Output CSV: {OUTPUT_CSV}")

# ── Cluster loop — one unit at a time ─────────────────────────────────────────
header_written = False

for unit_id in tqdm(unit_ids, desc=CLUSTER_LEVEL):

    # Load only this unit's rows from parquet
    if LEVEL_COL is None:
        df_unit = pd.read_parquet(SYNPOP_PARQUET)
    else:
        df_unit = pd.read_parquet(
            SYNPOP_PARQUET,
            filters=[(LEVEL_COL, '==', unit_id)]
        )

    if df_unit.empty:
        continue

    unit_rows = []

    if USE_GROUPED_CLUSTERING:
        # ── Grouped: split by employment status, cluster each group separately ──
        assigned = pd.Series(False, index=df_unit.index)

        # -- Proportional k: each group gets clusters proportional to its share --
        _gsizes = {}
        _seen   = pd.Series(False, index=df_unit.index)
        for _gname in GROUPS:
            _c = GROUP_COL.get(_gname)
            if _c == 'MISSING':
                continue
            if _c is not None and _c in df_unit.columns:
                _m = df_unit[_c] == 1.0
            elif _c is None:
                _m = ~_seen
            else:
                continue
            _gsizes[_gname] = int(_m.sum())
            _seen |= _m
        _total   = sum(_gsizes.values()) or 1
        _group_k = {g: max(1, round(MAX_TOTAL_CLUSTERS * n / _total)) for g, n in _gsizes.items()}

        for gname in GROUPS:
            col = GROUP_COL.get(gname)

            if col == "MISSING":
                continue
            if col is not None:
                mask = df_unit[col] == 1.0
            else:
                mask = ~assigned   # catch-all

            group_df = df_unit[mask].copy()
            if group_df.empty:
                continue
            assigned |= mask

            avail_feat = [
                c for c in feature_cols
                if c in group_df.columns and group_df[c].notna().any()
            ]
            if not avail_feat:
                continue

            coeffs     = normalise.fit(group_df, avail_feat)
            group_norm = normalise.apply(group_df, coeffs)

            n        = len(group_norm)
            max_k    = min(_group_k.get(gname, 1), max(1, n // MIN_CLUSTER_SIZE))
            chosen_k = cf.best_k_by_silhouette(group_norm[avail_feat].values, max_k)
            labels   = cf.fit_kmeans(group_norm[avail_feat].values, chosen_k)

            group_df = group_df.copy()
            group_df['_tribe_sub'] = labels

            for sub_id in sorted(group_df['_tribe_sub'].unique()):
                sub_df      = group_df[group_df['_tribe_sub'] == sub_id]
                tribe_label = f"{gname} {sub_id + 1}" if chosen_k > 1 else gname
                row = cf.build_dna_row(
                    tribe_label, sub_df, WAVE,
                    SUMMARY_VARS, VARIABLE_MAP, CATEGORICAL_VARS, CATEGORY_MAPS,
                    continuous_vars=CONTINUOUS_VARS,
                )
                row['unit_id']       = unit_id
                row['la_name']       = _la_name_map.get(unit_id, unit_id)
                row['cluster_level'] = CLUSTER_LEVEL
                row['group']         = gname
                unit_rows.append(row)

    else:
        # ── Flat: cluster whole unit population in one pass ───────────────────
        avail_feat = [
            c for c in feature_cols
            if c in df_unit.columns and df_unit[c].notna().any()
        ]
        if avail_feat:
            coeffs     = normalise.fit(df_unit, avail_feat)
            unit_norm  = normalise.apply(df_unit, coeffs)

            n        = len(unit_norm)
            max_k    = min(MAX_TOTAL_CLUSTERS, max(1, n // MIN_CLUSTER_SIZE))
            chosen_k = cf.best_k_by_silhouette(unit_norm[avail_feat].values, max_k)
            labels   = cf.fit_kmeans(unit_norm[avail_feat].values, chosen_k)

            df_unit = df_unit.copy()
            df_unit['_tribe_sub'] = labels

            for sub_id in sorted(df_unit['_tribe_sub'].unique()):
                sub_df      = df_unit[df_unit['_tribe_sub'] == sub_id]
                tribe_label = f"Cluster {sub_id + 1}"
                row = cf.build_dna_row(
                    tribe_label, sub_df, WAVE,
                    SUMMARY_VARS, VARIABLE_MAP, CATEGORICAL_VARS, CATEGORY_MAPS,
                    continuous_vars=CONTINUOUS_VARS,
                )
                row['unit_id']       = unit_id
                row['la_name']       = _la_name_map.get(unit_id, unit_id)
                row['cluster_level'] = CLUSTER_LEVEL
                row['group']         = "All"
                unit_rows.append(row)

    # Append this unit's rows to CSV
    if unit_rows:
        unit_df = pd.DataFrame(unit_rows)
        unit_df.to_csv(
            OUTPUT_CSV,
            mode='a',
            header=not header_written,
            index=False,
        )
        header_written = True

    del df_unit, unit_rows
    gc.collect()

print(f"\nDone. Results written to {OUTPUT_CSV}")
if OUTPUT_CSV.exists():
    result = pd.read_csv(OUTPUT_CSV)
    print(f"  {len(result)} tribe rows across {result['unit_id'].nunique()} units")
    print(result[['unit_id', 'cluster_level', 'group', 'tribe_label', 'size']].head(12).to_string(index=False))

    # Copy to API serving directory so the web app picks it up immediately
    api_clusters_dir = Path("../api/data/clusters")
    api_clusters_dir.mkdir(parents=True, exist_ok=True)
    import shutil
    api_copy = api_clusters_dir / OUTPUT_CSV.name
    shutil.copy2(OUTPUT_CSV, api_copy)
    print(f"  Copied to {api_copy}")


🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  
  FOUR-LA SUBSET ACTIVE — Newham, Tower Hamlets, Islington, Hounslow only (4 LAs)
  Set USE_FOUR_LA_SUBSET = False for all London or full UK.
🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  


🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  
  FOUR-LA SUBSET ACTIVE — Newham, Tower Hamlets, Islington, Hounslow only (4 LAs)
  Set USE_FOUR_LA_SUBSET = False for all London or full UK.
🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  

Removed existing LA_london_clusters.csv
Loaded 364 LA names from admin_geography_mappings.csv
Feature columns: 5 present, 0 missing

Grouped clustering: True

Group -> jbstat column (current-wave, mutually exclusive):
  Employed             -> o_jbstat_1
  Retired              -> o_jbstat_4
  Unemployed           -> o_jbstat_3
  Student              -> o_jbstat_7
  On leave            

LA: 100%|██████████| 4/4 [00:23<00:00,  5.82s/it]


Done. Results written to ../data/6_cluster/LA_london_clusters.csv
  55 tribe rows across 4 units
  unit_id cluster_level      group  tribe_label  size
E09000018            LA   Employed   Employed 1 17052
E09000018            LA   Employed   Employed 2 23927
E09000018            LA   Employed   Employed 3  7858
E09000018            LA   Employed   Employed 4 16445
E09000018            LA   Employed   Employed 5 11091
E09000018            LA   Employed   Employed 6  7819
E09000018            LA    Retired    Retired 1 18839
E09000018            LA    Retired    Retired 2  4064
E09000018            LA Unemployed Unemployed 1  4311
E09000018            LA Unemployed Unemployed 2  3727
E09000018            LA    Student      Student  3142
E09000018            LA   On leave     On leave  4720
  Copied to ../api/data/clusters/LA_london_clusters.csv
